In [ ]:
import json
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
OUTPUT_PATH = "./results/baseline.jsonl"

# load dataset
public_data = [json.loads(line) for line in open("./data/public.jsonl")]

n_mcq  = sum(bool(d.get("options")) for d in public_data)
n_free = sum(not d.get("options")   for d in public_data)
print(f"Loaded {len(public_data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

Loaded 1126 questions  (375 MCQ, 751 free-form)


In [2]:
# prompts
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Solve the problem carefully and briefly and put your final answer within \boxed{}."
    "If there are multiple answers, put them in a single \\boxed{} separated by commas."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Choose the single best answer from the options provided. "
    "Output only \\boxed{<letter>}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

In [3]:
# load model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=True,
    gpu_memory_utilization=0.88,
    max_model_len=30000, # 16384
    trust_remote_code=True,
    max_num_seqs=4, # 256
    max_num_batched_tokens=16384, # was 32768
)

sampling_params = SamplingParams(
    max_tokens=4096, # was 32768
    temperature=0,
)

print("Model loaded.")

INFO 04-09 21:00:44 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 30000, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.88, 'max_num_batched_tokens': 16384, 'max_num_seqs': 4, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 04-09 21:00:45 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 04-09 21:00:45 [model.py:1678] Using max model len 30000
INFO 04-09 21:00:45 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-09 21:00:45 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=25578) INFO 04-09 21:00:46 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat

(EngineCore pid=25578) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=25578) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(EngineCore pid=25578) INFO 04-09 21:00:48 [bitsandbytes_loader.py:786] Loading weights with BitsAndBytes quantization. May take a while ...


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=25578) INFO 04-09 21:00:50 [gpu_model_runner.py:4820] Model loading took 2.7 GiB memory and 1.999317 seconds
(EngineCore pid=25578) INFO 04-09 21:00:52 [backends.py:1051] Using cache directory: /home/lemon22/.cache/vllm/torch_compile_cache/b6b54c3bb9/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=25578) INFO 04-09 21:00:52 [backends.py:1111] Dynamo bytecode transform time: 1.74 s
(EngineCore pid=25578) INFO 04-09 21:00:53 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 1.266 s
(EngineCore pid=25578) INFO 04-09 21:00:53 [decorators.py:303] Directly load AOT compilation from path /home/lemon22/.cache/vllm/torch_compile_cache/torch_aot_compile/68dabb3cf45651214619bcb31e6a49b1a439fee535b949a4ba384e131a42e9e2/rank_0_0/model
(EngineCore pid=25578) INFO 04-09 21:00:53 [monitor.py:48] torch.compile took 3.23 s in total
(EngineCore pid=25578) INFO 04-09 21:00:53 [monitor.py:76] Initial profiling/warmup run took 

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 4/4 [00:00<00:00, 16.43it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 3/3 [00:00<00:00, 19.14it/s]


(EngineCore pid=25578) INFO 04-09 21:00:58 [gpu_model_runner.py:6046] Graph capturing finished in 1 secs, took 0.12 GiB
(EngineCore pid=25578) INFO 04-09 21:00:58 [gpu_worker.py:597] CUDA graph pool memory: 0.12 GiB (actual), 0.14 GiB (estimated), difference: 0.03 GiB (23.7%).
(EngineCore pid=25578) INFO 04-09 21:00:58 [core.py:283] init engine (profile, create kv cache, warmup model) took 8.69 seconds
Model loaded.


In [4]:
# Build prompts for first 5 entries
prompts = []
for item in public_data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

Generating responses for 5 questions...


Rendering prompts:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [5]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(public_data, responses), total=len(public_data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 5/1126 [00:00<00:16, 70.01it/s]

Scoring complete. 5 results.


In [6]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    1 /    2  (50.00%)
  Free-form  :    2 /    3  (66.67%)
  Overall    :    3 /    5  (60.00%)


In [7]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 5 records to output/baseline.jsonl
